# YOUR FIRST LAB
## Your first Frontier LLM Project

By the end of this course, you will have built an autonomous Agentic AI solution with 7 agents that collaborate to solve a business problem. All in good time! We will start with something smaller...

Our goal is to code a new kind of Web Browser. Give it a URL, and it will respond with a summary. The Reader's Digest of the internet!!

Welcome to the wonderful world of Data Science experimentation! Simply click in each "cell" with code in it, such as the cell immediately below this text, and hit Shift+Return to execute that cell. Be sure to run every cell, starting at the top, in order.


### Select the Kernel

Click on "Select Kernel" on the Top Right

Choose "Python Environments..."

Then choose the one that looks like `.venv (Python 3.12.x) .venv/bin/python` - it should be marked as "Recommended" and have a big star next to it.


### Note: you'll need to set the Kernel with every notebook..

In [3]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

# Connecting to OpenAI (or Ollama)

The next cell is where we load in the environment variables in your `.env` file and connect to OpenAI.  

If you'd like to use free Ollama instead, please see the README section "Free Alternative to Paid APIs", and if you're not sure how to do this, there's a full solution in the solutions folder (day1_with_ollama.ipynb).

In [4]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


# Let's make a quick call to a Frontier model to get started, as a preview!

In [5]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages


[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

In [6]:
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
response.choices[0].message.content

"Hi there! Nice to meet you. Welcome to our chat—I'm here to help with questions, explanations, writing, coding, planning, and more.\n\nWhat would you like to do today? If you’re not sure, here are a few quick ideas:\n- Ask for a simple explanation of a topic\n- Draft an email, message, or document\n- Brainstorm ideas for a project or story\n- Get step-by-step instructions for a task\n- Practice a new language or solve a math problem\n- Write or debug a bit of code\n\nTell me what you're curious about or what you’d like to work on."

## OK onwards with our first project

In [8]:
# Let's try out this utility

ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

Home - Edward Donner

Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 400,000 enrolled across 190

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [9]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [10]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```
To give you a preview, the next 2 cells make a rather simple call - we won't stretch the mighty GPT (yet!)

In [11]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
response.choices[0].message.content

'2 + 2 equals 4.'

## And now let's build useful messages for GPT-4.1-mini, using a function

In [12]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [14]:
# Try this out, and then try for a few more websites

messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nHome - Edward Donner\n\nHome\nAI Curriculum\nProficient AI Engineer\nConnect Four\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of\nNebula.io\n.

## Time to bring it together - the API for OpenAI is very simple!

In [15]:
# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [16]:
summarize("https://edwarddonner.com")

"# Edward Donner's Playground of AI Nerdom\n\nEd’s site is basically a shrine to his obsession with Large Language Models, coding, and barely competent electronic music production (he admits it, so it’s cute). He’s a big deal: CTO of Nebula.io, former startup founder with an acquisition under his belt, and surprisingly popular Udemy course creator with a mind-boggling 400,000 students worldwide.  \n\nThe site dishes out:  \n- A full AI curriculum for those who want to drink from Ed’s AI firehose  \n- Competitive LLM battles in games like Connect Four and “Outsmart” (Where bots get shady with each other)  \n- Ramblings and blog posts about all things AI and nerdy life stuff  \n\n**Recent News & Announcements:**  \n- Jan 4, 2026: Building AI and voice agents with n8n  \n- Nov 11, 2025: Recap of the electric vibe at a live AI event  \n- Sept 15, 2025: Launch of an AI Engineering MLOps track  \n- May 28, 2025: Guidance on course order for AI studies  \n\nBasically, if you want free-range E

In [17]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [18]:
display_summary("https://edwarddonner.com")

# Edward Donner's Corner of the Internet

Welcome to the shrine of Ed, the AI tinkerer who’s converted endless rambling about large language models into *bestselling* Udemy courses with 400,000 enthusiasts worldwide. He’s also a music hobbyist (very amateur, but hey, it counts) and a Hacker News nodder.

**What’s here?**  
- AI Curriculum for aspiring AI wizards  
- A “Proficient AI Engineer” track that sounds like it involves actual engineering, not just wizardry  
- “Connect Four” and “Outsmart,” where AI models duke it out in battles of wit and diplomacy — because even bots need drama  
- Frequent blog posts and newsletters for those who like their AI served fresh and snarky  

**News & Announcements** (aka Latest from Ed's AI Lab):  
- Jan 4, 2026: Build your own AI and voice agents with n8n. Fancy!  
- Nov 11, 2025: Ed revels in the "Unique Energy of an AI Live Event." Spoiler: it's probably a room full of code and caffeine.  
- Sept 15, 2025: Launching an AI Engineering MLOps track to get your AI models out of the basement and into the wild.  
- May 28, 2025: Tips on which order to binge-watch his AI courses — because AI education needs a Netflix-style guide apparently.  

**Contact & Socials:**  
Ed’s reachable via email if you want your inbox blessed with occasional wisdom bombs, and he's on all the usual LinkedIn, Twitter, and Facebook haunts.

In sum: Ed’s site is like that super nerdy friend who turned AI hype into an empire of courses and bot battles—and you’re invited to join the cult.

# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.

Websites that are rendered with Javascript, like React apps, won't show up. See the community-contributions folder for a Selenium implementation that gets around this. You'll need to read up on installing Selenium (ask ChatGPT!)

Also Websites protected with CloudFront (and similar) may give 403 errors - many thanks Andy J for pointing this out.

But many websites will work just fine!

In [19]:
display_summary("https://cnn.com")

# CNN: Your One-Stop Shop for News Overload

Welcome to CNN, where they cover *everything* from global conflicts (Ukraine-Russia, Israel-Hamas) to the latest in celebrity gossip, sports, science breakthroughs, and even your weather forecast—because who doesn’t need a side of climate crisis with their morning coffee?

### Breaking News & Announcements
- The site is *brimming* with up-to-the-minute news and live TV streams.
- They’re super eager for feedback, especially about ads. Expect pop-ups asking if ads annoyed you, froze your screen, or made the cat run away.
- With multiple editions (US, International, Arabic, Español), CNN is basically everywhere, like that clingy friend who follows you around but with more serious headlines.

### Categories Galore
Politics, business, tech, entertainment, health, style, travel, and yes, even a section dedicated to "Life, But Better" (because you obviously need it).

### Summary
If you want news bites, in every flavor imaginable, and a smorgasbord of videos, polls, and live updates, CNN is your chaotic, caffeinated fix. Just brace yourself for the relentless ad feedback requests—they really *care* if that video ad was "too loud."

In [20]:
display_summary("https://anthropic.com")

# Anthropic: AI with a Conscience

Welcome to Anthropic, where AI isn't just about flashy tech but also not blowing up the world. They’re a public benefit corporation swimming in high-minded ideals like safety, transparency, and responsible scaling. Their crown jewel? Claude Opus 4.5 — apparently the best model for coding, agents, and enterprise workflows. (Because who doesn’t want their AI to be both a genius coder and the office MVP?)

Newsflash: They just dropped Claude Opus 4.5, boasting advanced tool use on their developer platform. It’s like giving your AI superpowers but keeping it on a leash.

In summary: Anthropic builds AI that aims not to doom humanity but maybe just to help it flourish, all while juggling the thrills and chills of cutting-edge AI research. Safe, smart, and slightly earnest.

I need to add a part related to the setup
add explanation to the message and roles

Summary of the code

In [22]:
# Step 1: Create your prompts

system_prompt = "something here"
user_prompt = """
    Lots of text
    Can be pasted here
"""

# Step 2: Make the messages list

messages = [] # fill this in

# Step 3: Call OpenAI
# response =

# Step 4: print the result
# print(